In [1]:
import requests
from bs4 import BeautifulSoup

In [11]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import time
from selenium.webdriver.support.ui import WebDriverWait

In [12]:
chrome_options = Options()
chrome_options.add_argument("--headless")  # chạy nền, không mở cửa sổ
driver = webdriver.Chrome(options=chrome_options)

driver.get("https://leakbase.la/")  # URL trang forum
time.sleep(5)  # chờ JS load xong
wait = WebDriverWait(driver, 10)
threads = driver.find_elements(By.CSS_SELECTOR, "a.node-title")  # ví dụ class bài viết
for t in threads:
    print(t.text, t.get_attribute("href"))

driver.quit()

In [4]:
def scrape_website(url):
    try:
        # 1. Lấy HTML
        response = requests.get(url, timeout=10)
        response.raise_for_status()  # kiểm tra lỗi HTTP

        html = response.text

        # 2. Parse HTML bằng BeautifulSoup
        soup = BeautifulSoup(html, 'html.parser')

        # 3. Lấy text trong body
        content = soup.body.get_text(separator='\n', strip=True)

        print("=== Content Preview ===")
        print(content[:500])  # in 500 ký tự đầu
        return content

    except requests.exceptions.RequestException as e:
        print(f"Lỗi khi lấy content: {e}")
        return None


In [5]:
url = "https://leakbase.la/"
scrape_website(url)

=== Content Preview ===
Home
Unanswered threads
What's new
Members
Registered members
Current visitors
Upd FAQ
Payments
Log in
Register
Register
Menu
Log in
Register
Install the app
Install
Welcome guest!
There is no section for sales and data purchases on the forum. We do not make "advertising" and do not sell data. Before creating new topics and posts, please read the possible errors and rules
(In the "Help" section).
For advertising the telegram of communities, for the bases of Russia, for an attempt to sell anythin


'Home\nUnanswered threads\nWhat\'s new\nMembers\nRegistered members\nCurrent visitors\nUpd FAQ\nPayments\nLog in\nRegister\nRegister\nMenu\nLog in\nRegister\nInstall the app\nInstall\nWelcome guest!\nThere is no section for sales and data purchases on the forum. We do not make "advertising" and do not sell data. Before creating new topics and posts, please read the possible errors and rules\n(In the "Help" section).\nFor advertising the telegram of communities, for the bases of Russia, for an attempt to sell anything here: An account blocking will be issued.\nTop 5 DataLeak forums in 2024\n[According to journalists in the subject, the forum on such statistics. #2. LeakBase]\nRegister\nLogin\nJavaScript is disabled. For a better experience, please enable JavaScript in your browser before proceeding.\nYou are using an out of date browser. It  may not display this or other websites correctly.\nYou should upgrade or use an\nalternative browser\n.\nNew Messages\nNew Topics\nPremium Content\

In [13]:
import requests

url = 'https://leakbase.la/'

response = requests.get(url=url)

print(response.content)

b'<!DOCTYPE html>\n<html id="XF" lang="en-US" dir="LTR"\n\tdata-app="public"\n\tdata-template="forum_list"\n\tdata-container-key=""\n\tdata-content-key=""\n\tdata-logged-in="false"\n\tdata-cookie-prefix="xf_"\n\tdata-csrf="1765189482,110bba03d6d5ff1e44f6c853857e8422"\n\tclass="has-no-js template-forum_list"\n\t>\n<head>\n\t<meta charset="utf-8" />\n\t<meta http-equiv="X-UA-Compatible" content="IE=Edge" />\n\t<meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover">\n\n\t\n\t\n\t\n\n\t\n\t\t<title>LeakBase.la - Owned by Chucky</title>\n\t\n\n\t<link rel="manifest" href="/webmanifest.php">\n\t\n\t\t<meta name="theme-color" content="#fc4242" />\n\t\n\n\t<meta name="apple-mobile-web-app-title" content="LeakBase.la">\n\t\n\t\t<link rel="apple-touch-icon" href="/data/assets/logo/dreamstime_l_155219940-removebg-preview.png">\n\t\n\t\n\t\n\t\t\n\t\t<meta name="description" content="LeakBase.la - Official Community Forum" />\n\t\t<meta property="og:description" co

In [2]:
%cd ..

/home/ngocson/NSN/Project/cyber_intelligence_project


In [3]:
from src.detectors.rules_detector import DataLeakDetector

In [5]:
import csv 

In [6]:
def load_and_filter_csv(
    input_csv: str,
    output_csv: str = "vietnam_posts.csv"
):
    detector = DataLeakDetector()
    vietnam_rows = []

    with open(input_csv, "r", encoding="utf-8", errors="ignore") as f:
        reader = csv.DictReader(f)

        # Kiểm tra cột cần thiết
        if "title" not in reader.fieldnames or "content" not in reader.fieldnames:
            raise ValueError(
                f"CSV phải có cột 'title' và 'content'. "
                f"Hiện có: {reader.fieldnames}"
            )

        for idx, row in enumerate(reader, start=1):
            title = row.get("title", "")
            content = row.get("content", "")

            combined_text = f"{title} {content}"

            if detector.is_vietnam_post(combined_text):
                row["is_vietnam"] = "true"
                vietnam_rows.append(row)

    # Ghi ra file CSV mới
    if vietnam_rows:
        fieldnames = list(vietnam_rows[0].keys())

        with open(output_csv, "w", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(vietnam_rows)

    print(f"[DONE] Found {len(vietnam_rows)} Vietnam-related posts")
    print(f"[OUTPUT] Saved to {output_csv}")


# =====================================================
# RUN TEST
# =====================================================
if __name__ == "__main__":
    INPUT_FILE = "leakbase_output.csv"
    load_and_filter_csv(INPUT_FILE)

[DONE] Found 2 Vietnam-related posts
[OUTPUT] Saved to vietnam_posts.csv
